# Importing the library and file

In [1]:
import pandas as pd
import numpy as np
np.random.seed(42)

In [2]:
df = pd.read_csv("Raw_file.csv")
df

FileNotFoundError: [Errno 2] No such file or directory: 'PS_20174392719_1491204439457_log.csv'

# Sampling and Formatting the dataset. (This is Fact Dataset in the start schema)

In [ ]:
df.info()

df.describe()

df.head()

df.isnull().sum()

In [ ]:
df["type"].value_counts()                                                       #to see the payment types count


In [ ]:
df["isFraud"].value_counts(normalize=True)*100                                  #to see the if the fraud count is viable for sampling

In [ ]:
                                                                                #the fraud proportion is too low(.12) so random sampling might not be useful for the analysis so we'll take different approach for sampling

fraud_df = df[df["isFraud"] == 1]                                               #splitting the dataset is fraud and non fraud part
nonfraud_df = df[df["isFraud"] == 0]

In [ ]:
fraud_df = fraud_df.copy()                                                      #too safekeep the fraud case thats why different cell

In [ ]:
target_size = 60000                                                             #Total sample size

nonfraud_sample = nonfraud_df.sample(
    n=target_size - len(fraud_df),
    random_state=42
)                                                                               #sampling the non_fraud data with sampling type 42 so that data can be reproducible

In [ ]:
df_sample = pd.concat(
    [fraud_df, nonfraud_sample],
    ignore_index=True
)                                                                               #combibining the dataset of fraud and non_fraud

In [ ]:
df_sample = df_sample.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)                                                        #shuffling the data to avoid the grouping and bias

In [ ]:
df_sample["isFraud"].value_counts(normalize=True)*100                           #now the fraud_cases are viable ~13% so we can move on with this data

In [ ]:
df_sample["TransactionID"] = [
    f"TXN{i:06d}" for i in range(1, len(df_sample) + 1)
]                                                                               #adding the unique transaction for the dataset which works as primary key for this data set

cols = ["TransactionID"] + [c for c in df_sample.columns if c != "TransactionID"]
df_sample = df_sample[cols]                                                     #moving the TransactionID column to the front

In [ ]:
df_sample

In [ ]:
bins = [-1,1000,10000,50000,float("inf")]                                       #categorising the transaction amount for better visual analysis. We didnt use if here cause if in
                                                                                #future we want to add new value in labels we can just update bins and labels instead of whole function
                                                                                #bins probably start at 0, and there are transactions with amount = 0 or values outside the bin edges. so started the bin range with -1

labels = [
    "Small",
    "Medium",
    "Large",
    "Very Large"
]

df_sample["AmountBand"] = pd.cut(
    df_sample["amount"],
    bins=bins,
    labels=labels
)

In [ ]:
fee_rate = {                                                                    #transaction fees as the only source of revenue for the company
    "PAYMENT":0.003,
    "TRANSFER":0.004,
    "CASH_OUT":0.002,
    "CASH_IN":0.0015,
    "DEBIT":0.0025
}

df_sample["Fee"] = (
    df_sample["amount"] *
    df_sample["type"].map(fee_rate)
).round(2)

In [ ]:
df_sample["Cashback"] = 0.0                                                     #cashback system for the payment type of transaction. Used flot type to avoid future variable type erros

mask = df_sample["type"]=="PAYMENT"

df_sample.loc[mask,"Cashback"] = (
    df_sample.loc[mask,"amount"]*0.005
).clip(upper=100)

In [ ]:
df_sample["NetRevenue"] = (                                                     #net revenue calculation
    df_sample["Fee"] -
    df_sample["Cashback"]
).round(2)

In [ ]:
df_sample["HighValue"] = (                                                      #high value transaction calculation as completely distiguised table as its a important KPI
    df_sample["amount"]>50000
)

In [ ]:
df_sample["BalanceReduction"] = (                                               #balance reduction calculation
    df_sample["oldbalanceOrg"]-
    df_sample["newbalanceOrig"]
)

In [ ]:
df_sample.info()

In [ ]:
df_sample.to_csv(                                                               #exporting the dataset
    "Fact_Transactions.csv",
    index=False
)

# Creating the Customer Dimension Tables


In [ ]:
num_customers = 20000                                                           #generating the customersID cause the customer ID in the transaction table are all unique and is not useful for customer analysis

dim_customer = pd.DataFrame({
    "CustomerKey": range(1, num_customers + 1),
    "CustomerID": [f"CUST{str(i).zfill(5)}" for i in range(1, num_customers + 1)]
})

dim_customer.head()

In [ ]:
segments = ["Standard", "Gold", "Premium"]                                      #customer segment generation
segment_prob = [0.65, 0.25, 0.10]

dim_customer["CustomerSegment"] = np.random.choice(
    segments,
    size=num_customers,
    p=segment_prob
)

In [ ]:
states = [                                                                      #limiting the fintech market to some states only
    "Maharashtra",
    "Karnataka",
    "Delhi",
    "Tamil Nadu",
    "Gujarat",
    "Telangana",
    "West Bengal",
    "Uttar Pradesh",
    "Kerala",
    "Rajasthan"
]

state_prob = [
    0.18,
    0.12,
    0.10,
    0.09,
    0.08,
    0.08,
    0.08,
    0.10,
    0.08,
    0.09
]

dim_customer["State"] = np.random.choice(
    states,
    size=num_customers,
    p=state_prob
)

In [ ]:
city_dict = {                                                                   #assigning the city districts
    "Maharashtra": ["Mumbai", "Pune", "Nagpur"],
    "Karnataka": ["Bengaluru", "Mysuru", "Mangaluru"],
    "Delhi": ["New Delhi"],
    "Tamil Nadu": ["Chennai", "Coimbatore", "Madurai"],
    "Gujarat": ["Ahmedabad", "Surat", "Vadodara"],
    "Telangana": ["Hyderabad", "Warangal"],
    "West Bengal": ["Kolkata", "Siliguri", "Durgapur"],
    "Uttar Pradesh": ["Lucknow", "Noida", "Kanpur"],
    "Kerala": ["Kochi", "Thiruvananthapuram", "Kozhikode"],
    "Rajasthan": ["Jaipur", "Jodhpur", "Udaipur"]
}

dim_customer["City"] = dim_customer["State"].apply(
    lambda x: np.random.choice(city_dict[x])
)

In [ ]:
gender = ["Male", "Female"]                                                     #assigning the genders

gender_prob = [0.58, 0.42]

dim_customer["Gender"] = np.random.choice(
    gender,
    size=num_customers,
    p=gender_prob
)

In [ ]:
age_groups = [                                                                  #assigning the age groups
    "18-25",
    "26-35",
    "36-45",
    "46-60",
    "60+"
]

age_prob = [
    0.22,
    0.36,
    0.24,
    0.13,
    0.05
]

dim_customer["AgeGroup"] = np.random.choice(
    age_groups,
    size=num_customers,
    p=age_prob
)

In [ ]:
income = [                                                                      #assigning the income group
    "Low",
    "Middle",
    "High"
]

income_prob = [
    0.35,
    0.50,
    0.15
]

dim_customer["IncomeGroup"] = np.random.choice(
    income,
    size=num_customers,
    p=income_prob
)

In [ ]:
kyc = [                                                                         #assigning the KYC status
    "Verified",
    "Pending",
    "Rejected"
]

kyc_prob = [
    0.96,
    0.03,
    0.01
]

dim_customer["KYCStatus"] = np.random.choice(
    kyc,
    size=num_customers,
    p=kyc_prob
)

In [ ]:
join_years = [                                                                  #assigning the join years
    2019,
    2020,
    2021,
    2022,
    2023,
    2024,
    2025,
    2026
]

join_prob = [
    0.04,
    0.06,
    0.10,
    0.12,
    0.14,
    0.16,
    0.18,
    0.20
]

dim_customer["JoinYear"] = np.random.choice(
    join_years,
    size=num_customers,
    p=join_prob
)

In [ ]:
dim_customer["CustomerTenure"] = 2026 - dim_customer["JoinYear"]                #assigning the customer tenure by year

In [ ]:
dim_customer.tail()

In [ ]:
dim_customer.to_csv(                                                            #exporting the dataset
    "dim_customer.csv",
    index=False
)

In [ ]:
from google.colab import files
files.download('dim_customer.csv')

# Replacing the customerID to the transaction table with the new customer ID

In [ ]:
customer_weights = (                                                            #assigning the customer segments a weightage to map to the transaction table
    dim_customer["CustomerSegment"]                                             #so that the high value transaction can be mapped to gold or premium customers
    .map({
        "Standard": 1,
        "Gold": 2,
        "Premium": 4
    })
)

In [ ]:
df_sample["CustomerKey"] = np.random.choice(                                    #replacing the old customerID column with the new one
    dim_customer["CustomerKey"],                                                #mapping the CustomerKey and not CustomerId as if the naming convention changes, the customer table will still be mapped
    size=len(df_sample),
    replace=True,
    p=customer_weights / customer_weights.sum()
)

df_sample.drop(columns=["nameOrig"], inplace=True)


#Creating the Merchant Dimension Tables

In [ ]:
num_merchants = 8000                                                            #creating the merchant table to assign all the merchant attributes

dim_merchant = pd.DataFrame({
    "MerchantKey": range(1, num_merchants + 1),
    "MerchantID": [f"MER{str(i).zfill(5)}" for i in range(1, num_merchants + 1)]
})

dim_merchant.head()

In [ ]:
merchant_categories = [                                                         #assigning the merchant categories
    "Retail",
    "E-Commerce",
    "Food & Dining",
    "Fuel",
    "Healthcare",
    "Travel",
    "Entertainment",
    "Education",
    "Utilities",
    "Government"
]

category_prob = [
    0.22,
    0.18,
    0.15,
    0.08,
    0.08,
    0.07,
    0.06,
    0.05,
    0.07,
    0.04
]

dim_merchant["MerchantCategory"] = np.random.choice(
    merchant_categories,
    size=num_merchants,
    p=category_prob
)

In [ ]:
business_size = [                                                               #assigning the business size for the merchants
    "Small",
    "Medium",
    "Enterprise"
]

size_prob = [
    0.65,
    0.25,
    0.10
]

dim_merchant["BusinessSize"] = np.random.choice(
    business_size,
    size=num_merchants,
    p=size_prob
)

In [ ]:
states = [                                                                      #limiting the fintech market to some states only
    "Maharashtra",
    "Karnataka",
    "Delhi",
    "Tamil Nadu",
    "Gujarat",
    "Telangana",
    "West Bengal",
    "Uttar Pradesh",
    "Kerala",
    "Rajasthan"
]

state_prob = [
    0.18,
    0.12,
    0.10,
    0.09,
    0.08,
    0.08,
    0.08,
    0.10,
    0.08,
    0.09
]

dim_merchant["State"] = np.random.choice(
    states,
    size=num_merchants,
    p=state_prob
)

In [ ]:
city_dict = {                                                                   #assigning the city to the merchants
    "Maharashtra": ["Mumbai", "Pune", "Nagpur"],
    "Karnataka": ["Bengaluru", "Mysuru", "Mangaluru"],
    "Delhi": ["New Delhi"],
    "Tamil Nadu": ["Chennai", "Coimbatore", "Madurai"],
    "Gujarat": ["Ahmedabad", "Surat", "Vadodara"],
    "Telangana": ["Hyderabad", "Warangal"],
    "West Bengal": ["Kolkata", "Siliguri", "Durgapur"],
    "Uttar Pradesh": ["Lucknow", "Noida", "Kanpur"],
    "Kerala": ["Kochi", "Thiruvananthapuram", "Kozhikode"],
    "Rajasthan": ["Jaipur", "Jodhpur", "Udaipur"]
}

dim_merchant["City"] = dim_merchant["State"].apply(
    lambda x: np.random.choice(city_dict[x])
)

In [ ]:
settlement = [                                                                  #assigning the settlement time
    "T+1",
    "T+2",
    "T+3"
]

settlement_prob = [
    0.70,
    0.25,
    0.05
]

dim_merchant["SettlementCycle"] = np.random.choice(
    settlement,
    size=num_merchants,
    p=settlement_prob
)

In [ ]:
partner_year = [                                                                #assigning the year from when the merchants are added to the platform
    2019,
    2020,
    2021,
    2022,
    2023,
    2024,
    2025,
    2026
]

partner_prob = [
    0.04,
    0.06,
    0.10,
    0.12,
    0.13,
    0.17,
    0.19,
    0.19
]

dim_merchant["PartnerSince"] = np.random.choice(
    partner_year,
    size=num_merchants,
    p=partner_prob
)

In [ ]:
dim_merchant["MerchantTenure"] = (                                              #calculating the merchant tenure to current year
    2026 - dim_merchant["PartnerSince"]
)

In [ ]:
dim_merchant.head()

In [ ]:
dim_merchant.to_csv(                                                            #exporting the dataset
    "dim_merchant.csv",
    index=False
)

# Replacing the merchantID to the transaction table with the new merchantID

In [ ]:
merchant_weights = (
    dim_merchant["BusinessSize"]
    .map({
        "Small": 1,
        "Medium": 3,
        "Enterprise": 8
    })
)

In [ ]:
df_sample["MerchantKey"] = np.random.choice(
    dim_merchant["MerchantKey"],
    size=len(df_sample),
    replace=True,
    p=merchant_weights / merchant_weights.sum()
)

In [ ]:
df_sample.drop(columns=["nameDest"], inplace=True)

In [ ]:
df_sample.head()

# Creating Date Dimension


In [ ]:
start_date = pd.Timestamp("2026-01-01 00:00:00")                                #creating the time stamp for the transaction

dim_date = pd.DataFrame({
    "step": range(1, num_steps + 1)
})

dim_date["DateTime"] = (
    start_date +
    pd.to_timedelta(dim_date["step"] - 1, unit="h")
)

In [ ]:
dim_date["Date"] = dim_date["DateTime"].dt.date                                 #assigning the Date, Year, Month, Quarter, Weekday, Day, Hour according to the timestamp
dim_date["Year"] = dim_date["DateTime"].dt.year
dim_date["Month"] = dim_date["DateTime"].dt.month_name()
dim_date["MonthNo"] = dim_date["DateTime"].dt.month
dim_date["Quarter"] = dim_date["DateTime"].dt.quarter
dim_date["Weekday"] = dim_date["DateTime"].dt.day_name()
dim_date["Day"] = dim_date["DateTime"].dt.day
dim_date["Hour"] = dim_date["DateTime"].dt.hour

In [ ]:
dim_date["Weekend"] = np.where(                                                 #assigning the weekend and weekday
    dim_date["Weekday"].isin(["Saturday", "Sunday"]),
    "Weekend",
    "Weekday"
)

In [ ]:
def time_bucket(hour):                                                          #assigning the time bucket according to the hour
    if 0 <= hour < 6:
        return "Night"
    elif 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 18:
        return "Afternoon"
    else:
        return "Evening"

dim_date["TimeBucket"] = dim_date["Hour"].apply(time_bucket)

In [ ]:
dim_date["DateKey"] = (                                                         #assigning the date key
    dim_date["DateTime"]
    .dt.strftime("%Y%m%d%H")
    .astype(int)
)

In [ ]:
dim_date = dim_date[                                                            #rearranging the date table
    [
        "DateKey",
        "step",
        "DateTime",
        "Date",
        "Year",
        "Quarter",
        "MonthNo",
        "Month",
        "Day",
        "Weekday",
        "Weekend",
        "Hour",
        "TimeBucket"
    ]
]

In [ ]:
dim_date.to_csv(                                                                #exporting the dataset
    "dim_date.csv",
    index=False
)